# AGENTIC AI
## SESSION 17

Assignment questions and answers


## Question 1

Set up a simple MCP (Multi-Channel Processing) server using Python and SQLite that listens for incoming requests on a specified port and can handle basic commands to add or retrieve user data.

**Answer:**

```python
import json
import sqlite3
from http.server import BaseHTTPRequestHandler, HTTPServer

PORT = 8080

# Initialize SQLite database
conn = sqlite3.connect("mcp_data.db", check_same_thread=False)
cursor = conn.cursor()
cursor.execute(
    "CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT)"
)
conn.commit()


class MCPHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length))
        action = data.get("action")
        response = {}

        if action == "add":
            cursor.execute("INSERT INTO users (name) VALUES (?)", (data["name"],))
            conn.commit()
            response = {
                "status": "success",
                "message": f"User {data['name']} added.",
            }

        elif action == "get":
            cursor.execute("SELECT * FROM users")
            users = cursor.fetchall()
            response = {"status": "success", "users": users}

        else:
            response = {"status": "error", "message": "Invalid command"}

        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(response).encode())


if __name__ == "__main__":
    server = HTTPServer(("0.0.0.0", PORT), MCPHandler)
    print(f"MCP Server running on port {PORT}")
    server.serve_forever()
```

## Question 2

Design a function store_order(order_id, user_id, amount) that inserts a new order into an 'orders' table in your MCP server's SQLite database, then returns a success message.

Hint: Use parameterized SQL queries to prevent SQL injection.

**Answer:**

```python
import sqlite3


def store_order(order_id: int, user_id: int, amount: float) -> str:
    conn = sqlite3.connect("mcp_database.db")
    cursor = conn.cursor()

    query = (
        "INSERT INTO orders (order_id, user_id, amount) VALUES (?, ?, ?)"
    )
    cursor.execute(query, (order_id, user_id, amount))

    conn.commit()
    conn.close()

    return f"Order {order_id} successfully stored."
```

## Question 3

Implement a handler in your MCP server to process a 'GET_USER_ORDERS' command, which takes a user_id and returns all order_ids and amounts for that user from the database.

**Answer:**

```python
@mcp.tool()
def get_user_orders(user_id: int) -> list[dict]:
    """Process GET_USER_ORDERS command to fetch user order IDs and amounts."""
    cursor.execute(
        "SELECT order_id, amount FROM orders WHERE user_id = ?", (user_id,)
    )
    results = cursor.fetchall()
    return [{"order_id": row[0], "amount": row[1]} for row in results]
```

## Question 4

Simulate a Zomato-style order history: extend your MCP server so it can handle a 'GET_LAST_N_ORDERS' command, returning the N most recent orders for a user, sorted by order time.

Constraint: Add a timestamp column to your orders table and use it for sorting.

**Answer:**

```python
import sqlite3
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ZomatoOrderServer")

# Database setup with timestamp column
def init_db():
    conn = sqlite3.connect("zomato.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            order_id INTEGER PRIMARY KEY,
            user_id TEXT,
            items TEXT,
            total_amount REAL,
            order_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

init_db()

@mcp.tool(name="GET_LAST_N_ORDERS")
def get_last_n_orders(user_id: str, n: int) -> list[dict]:
    """Returns the N most recent orders for a user, sorted by order time."""
    conn = sqlite3.connect("zomato.db")
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT order_id, items, total_amount, order_time 
        FROM orders 
        WHERE user_id = ? 
        ORDER BY order_time DESC 
        LIMIT ?
    """, (user_id, n))
    
    rows = cursor.fetchall()
    conn.close()
    
    return [
        {
            "order_id": row[0],
            "items": row[1],
            "total_amount": row[2],
            "order_time": row[3]
        }
        for row in rows
    ]

if __name__ == "__main__":
    mcp.run()
```

## Question 5

Use ChatGPT or Copilot to generate a Python function that deletes an order by order_id from your MCP server database, then test and refine the generated code if needed.

**Answer:**

```python
import sqlite3
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("OrderManager")
DB_PATH = "orders.db"


@mcp.tool()
def delete_order(order_id: int) -> str:
    """Delete an order by its order_id from the database."""
    try:
        with sqlite3.connect(DB_PATH) as conn:
            cursor = conn.cursor()
            cursor.execute(
                "DELETE FROM orders WHERE order_id = ?", (order_id,)
            )

            if cursor.rowcount == 0:
                return f"Order {order_id} not found."

            conn.commit()
            return f"Order {order_id} successfully deleted."
    except sqlite3.Error as e:
        return f"Database error: {str(e)}"
```